# 15.1 Dense 的計算方式是什麼？


一句話有兩個token，另一句有四個token，模型需要兩倍的權重（weight）嗎？不需要：兩句用的是同一組權重，差別只在要算幾次。要看清這件事，得分開「儲存規則」與「使用規則」。token是切詞（tokenization，[6.1](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html)）得到的單位，每個token進入模型時是一列特徵數字。在[一層block](https://birdhackor.github.io/tiny-perceptron-vlm/4.5.html)裡，注意力讓不同位置交換訊息，接著每個位置各自通過同一個小型[前饋網路](https://birdhackor.github.io/tiny-perceptron-vlm/4.4.html)（Feed-Forward Network，FFN）。本節的材料是一張二列、每列兩項特徵的表；增加的是輸入列，處理它的權重先不改。

Dense在本章指每個token都使用同一組完整FFN，沒有先挑選某幾組權重。例如FFN先把兩個特徵擴成三個，經過啟用函式（activation function，[2.4](https://birdhackor.github.io/tiny-perceptron-vlm/2.4.html)），再收回兩個。第一張表有 `2×3=6` 個權重，第二張有 `3×2=6` 個，共12個。不論處理兩列還是四列，這兩張表都不會多一份；但四列要做四次相同種類的計算，因此工作量比兩列多。共享參數（parameter）意味著用同一規則，並不是所有位置只計算一次。

下列程式直接指定兩張表，不需訓練就能追蹤輸出。`torch.relu`就是2.4的ReLU：負值改成零、正值保留，放在兩次矩陣乘法之間當啟用函式。這段計算相當於[4.4](https://birdhackor.github.io/tiny-perceptron-vlm/4.4.html)的 `DenseFFN(2, hidden=3)`，只是為了手算簡化了兩處：啟用函式用最直接的ReLU，`DenseFFN` 用的是GELU；兩張表不加偏移（bias），`DenseFFN` 的 `nn.Linear` 則帶偏移。不加偏移，參數數量才恰好是上面的12個。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

up = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])
down = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])


def ffn(x):
    return torch.relu(x @ up) @ down


x = torch.tensor([[1.0, 2.0], [2.0, 1.0]])
for batch in (x, torch.cat([x, x], dim=0)):
    print("輸出", ffn(batch).tolist())
    print("token數", len(batch), "權重數", up.numel() + down.numel())

`@`的規則是每列輸入對上權重表的每一欄，同位置先相乘、再相加，產生該欄的一個結果。第一列 `[1,2]`乘up時，三欄分別算 `1×1+2×0=1`、`1×0+2×1=2`、`1×1+2×1=3`，得到 `[1,2,3]`。都為正所以relu不改它；再乘down，兩欄分別算 `1×1+2×0+3×1=4`、`1×0+2×1+3×1=5`，得到 `[4,5]`。第二列用同一規則得到 `[5,4]`。第一次輸出是這兩列，第二次則重複兩遍；兩次印出的權重數都為12。`torch.cat`沿列方向把輸入接起來，沒有複製模型；`numel`只數表中的元素，不數輸出與中間特徵（activation）。

後面的MoE會準備多組FFN，讓每個token只選其中少數組。現在這個Dense規則是比較基準：要知道MoE省掉什麼，先知道Dense對每列完整計算了哪些東西。不能從「Dense只存一套」斷定它一定用較少計算，也不能從token增加推出參數跟著增加；輸入長度、參數容量、工作量是三個不同量。

練習把 `torch.cat([x, x], dim=0)` 改成 `torch.cat([x, x, x], dim=0)`，讓第二輪的兩個輸入各重複三遍。執行前預測第二輪印出六列輸出、token數6，權重數仍是12。再把 `x` 的第一列 `[1.0, 2.0]` 改成 `[1.0, -2.0]`，先手算：乘up得 `[1,-2,-1]`，ReLU把兩個負值改成零得 `[1,0,0]`，再乘down得 `[1,0]`；執行核對第一列輸出。若增加的只是列數，FFN的每列規則保持不變；若要讓FFN變寬，則必須連兩張權重表的形狀一起改。

<details>
<summary>選讀：來源、量測條件與實驗紀錄</summary>

本章後面用TinyStories短篇故事做了真正更新參數的Dense／MoE比較，其中也有這樣一個Dense基準：特徵寬度（width）64、兩層，每層FFN中間寬度（hidden）256，連偏移共 `64×256+256+256×64+64=33,088` 個參數，整個模型141,568個參數。它更新180次後，51篇驗證故事的代價（loss）2.29999，52篇測試故事2.32093；這不是上面12個權重的玩具程式。數字出自[MoE實驗紀錄（JSON）](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/moe.json)，這是機器可讀的完整結果：看 `results.variants.dense_active_top1`，`model.parameters` 是參數總數，`heldout.validation.nll` 與 `heldout.test.nll` 是驗證與測試代價。重做命令是[T.8固定MoE配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#fixed-moe-recipe)裡的 `--experiment moe`。全部七個模型的代價與有效目標數在[15.13](https://birdhackor.github.io/tiny-perceptron-vlm/15.13.html)並排。

</details>
